In [ ]:
import numpy as np
from dvr2d import *
from utils import *
import matplotlib.pyplot as plt

from qiskit import Aer
from qiskit.algorithms import VQE
from qiskit.algorithms.optimizers import COBYLA, L_BFGS_B, SLSQP, BOBYQA, NELDER_MEAD, SNOBFIT
from qiskit.circuit.library import TwoLocal
from qiskit.utils import QuantumInstance, algorithm_globals
from qiskit.quantum_info.operators import Operator
from qiskit.opflow import PrimitiveOp
%matplotlib inline

plt.rcParams["figure.figsize"] = (8,6)
plt.rcParams["font.size"] = 14
plt.rcParams["figure.dpi"] = 300

%load_ext autoreload

In [ ]:
r = []
costheta = []
v = []
with open('arhcl.txt', 'r') as f:
# with open('h63.txt', 'r') as f:
    lines = f.readlines()
    for l in lines:
        r.append(float(l[:22]))
        costheta.append(float(l[22:53]))
        v.append(float(l[53:-1]))

r = np.array(r)
costheta = np.array(costheta)
theta = np.arccos(costheta)
v = np.array(v)
print(v.shape)
# print(theta)
print(np.unique(r))
print(np.unique(theta))
print(np.unique(r)[130])
print(v)

In [ ]:
%autoreload
pot2d = interp_pot2d(r, costheta, v)

In [ ]:
%autoreload
N_R = 35
N_theta = 4
l = 1
K_max = 0
r_min = 3.4
r_max = 5

Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, 0, l=l, r_min=r_min, r_max=r_max)
Rs_angs = Rs_DVR * arhcl_params['S'] * au_to_angs
print(Rs_angs)
print(Rs_angs.shape)
N_R_lim = Rs_angs.shape[0]

h_dvr = get_ham_DVR(pot2d, N_R, N_theta, 0, 0, l=l, r_min=r_min, r_max=r_max, trunc=0) * hartree
print(h_dvr.shape)
eigvals, eigvecs = np.linalg.eigh(h_dvr)
print(eigvals[0], eigvecs.T[0])
# plt.bar(range(eigvecs.shape[1]), eigvecs.T[0])

In [ ]:
cmap = plt.get_cmap('tab10')
theta_p, r_p, v_p = theta[v < 0], r[v < 0], v[v < 0]
coef = 180 / np.pi

p = plt.tricontour(theta_p[r_p < 5.5] * coef, r_p[r_p < 5.5], v_p[r_p < 5.5], levels=np.linspace(-175, 0, 26), colors='black', linestyles='solid', alpha=0.8)
# plt.plot(theta[ind_min1], r[ind_min1], 'x', c='black')
# plt.plot(theta[ind_min2], r[ind_min2], 'x', c='black')
# plt.plot(1.8754889808102941, 3.897, 'o', c='black')
Rs_grid, Xs_grid = np.meshgrid(Rs_DVR * arhcl_params['S'] * au_to_angs, np.arccos(Xs_K[0]))
Rs_grid, Xs_grid = Rs_grid.flatten(), Xs_grid.flatten()

labels = p.levels[:-15:]
plt.clabel(p, inline=1, fontsize=5, levels=labels, fmt='%.2f', use_clabeltext=True)
plt.ylim()
plt.xlabel('$\\theta (\degree)$')
plt.ylabel('$R$ (A)')
plt.ylim((3, 5.5))
# plt.xlim((0, np.pi))

# offset = 0.05
# plt.bar(np.arccos(Xs_K[0]), eigvecs5.T[0][::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 1 * offset, eigvecs5.T[0][1::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 2 * offset, eigvecs5.T[0][2::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 3 * offset, eigvecs5.T[0][3::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 4 * offset, eigvecs5.T[0][4::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.axhline(4.0, linestyle='dashed', c='black', linewidth=0.2)


for i in range(N_theta):
    evec = eigvecs.T[0][i * N_R_lim:(i+1) * N_R_lim]
    epos = np.where(evec >=0, evec, 0)
    eneg = np.where(evec <0, -evec, 0)
    plt.scatter(np.repeat(np.arccos(Xs_K[0][i]) * coef, N_R_lim), Rs_DVR * arhcl_params['S'] * au_to_angs, marker='o', c='black', zorder=20, alpha=0.7)
    # plt.scatter(np.repeat(np.arccos(Xs_K[0][i]), N_R_lim), Rs_DVR * arhcl_params['S'] * au_to_angs, marker='o', facecolors='black', edgecolors='black', s=epos * 400, zorder=20, alpha=0.7)
    # plt.scatter(np.repeat(np.arccos(Xs_K[0][i]), N_R_lim), Rs_DVR * arhcl_params['S'] * au_to_angs, facecolors='none', edgecolors='black', s=eneg * 400, alpha=0.7, zorder=20)

In [ ]:
h_dvr_pauli = pauli_decompose(h_dvr)

np.allclose(h_dvr, h_dvr_pauli.to_matrix())

print(h_dvr.shape)
print(h_dvr_pauli.coeffs.shape)
# print(h_dvr_pauli)

In [ ]:
ansatz1 = TwoLocal(5, rotation_blocks=['ry'], entanglement_blocks='cx', entanglement='full', reps=1).decompose()
print(ansatz1.depth())
ansatz1.draw(output='mpl', fold=-1)
# print(ansatz)

In [ ]:
optimizers = [COBYLA(maxiter=8000), L_BFGS_B(maxfun=8000), SLSQP(maxiter=1000), NELDER_MEAD(maxfev=8000)]
# from qiskit.algorithms.optimizers import NELDER_MEAD
# optimizers = [SLSQP(maxiter=600)]
converge_cnts1 = np.empty([len(optimizers)], dtype=object)
converge_vals1 = np.empty([len(optimizers)], dtype=object)

repeat = 3
params = None
# params = np.array([0.0 for i in range(ansatz1.num_parameters)])
for i, optimizer in enumerate(optimizers):
    print('Optimizer: {}        '.format(type(optimizer).__name__))
    algorithm_globals.random_seed = 42

    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        print(f'\r{eval_count}, {mean}', end='')
    
    best_res1 = None
    for j in range(repeat):
        counts = []
        values = []
        vqe = VQE(ansatz1, optimizer, callback=store_intermediate_result, initial_point=params, 
                quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
        result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
        print()

        if (best_res1 is None) or (values[-1] < best_res1):
            best_res1 = values[-1]
            converge_cnts1[i] = np.asarray(counts)
            converge_vals1[i] = np.asarray(values)
print('\nOptimization complete      ')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts1, converge_vals1)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals1[i][-1], abs(ref - converge_vals1[i][-1]))
    pylab.plot(converge_cnts1[i], converge_vals1[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals1[:, -1]))

In [ ]:
ansatz2 = TwoLocal(5, rotation_blocks=['ry'], entanglement_blocks='cx', entanglement='full', reps=2).decompose()
print(ansatz2.depth())
ansatz2.draw(output='mpl', fold=-1)

In [ ]:
optimizers = [COBYLA(maxiter=8000), L_BFGS_B(maxfun=8000), SLSQP(maxiter=1000), NELDER_MEAD(maxfev=8000)]
# from qiskit.algorithms.optimizers import NELDER_MEAD
# optimizers = [SLSQP(maxiter=600)]
converge_cnts2 = np.empty([len(optimizers)], dtype=object)
converge_vals2 = np.empty([len(optimizers)], dtype=object)

repeat = 3
params = None
# params = np.array([0.0 for i in range(ansatz1.num_parameters)])
for i, optimizer in enumerate(optimizers):
    print('Optimizer: {}        '.format(type(optimizer).__name__))
    algorithm_globals.random_seed = 42

    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        print(f'\r{eval_count}, {mean}', end='')
    
    best_res2 = None
    for j in range(repeat):
        counts = []
        values = []
        vqe = VQE(ansatz2, optimizer, callback=store_intermediate_result, initial_point=params, 
                quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
        result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
        print()

        if (best_res2 is None) or (values[-1] < best_res2):
            best_res2 = values[-1]
            converge_cnts2[i] = np.asarray(counts)
            converge_vals2[i] = np.asarray(values)
print('\nOptimization complete      ')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts2, converge_vals2)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals2[i][-1], abs(ref - converge_vals2[i][-1]))
    pylab.plot(converge_cnts2[i], converge_vals2[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts2[i][:], abs(ref - converge_vals2[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts2[i][:], abs(ref - converge_vals2[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals2[:, -1]))

In [ ]:
ansatz3 = TwoLocal(5, rotation_blocks=['ry'], entanglement_blocks='cx', entanglement='linear', reps=4).decompose()
print(ansatz3.depth())
ansatz3.draw(output='mpl', fold=-1)

In [ ]:
optimizers = [COBYLA(maxiter=8000), L_BFGS_B(maxfun=8000), SLSQP(maxiter=1000), NELDER_MEAD(maxfev=8000)]
# from qiskit.algorithms.optimizers import NELDER_MEAD
# optimizers = [SLSQP(maxiter=600)]
converge_cnts3 = np.empty([len(optimizers)], dtype=object)
converge_vals3 = np.empty([len(optimizers)], dtype=object)

repeat = 3
params = None
# params = np.array([0.0 for i in range(ansatz1.num_parameters)])
for i, optimizer in enumerate(optimizers):
    print('Optimizer: {}        '.format(type(optimizer).__name__))
    algorithm_globals.random_seed = 42

    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        print(f'\r{eval_count}, {mean}', end='')
    
    best_res3 = None
    for j in range(repeat):
        counts = []
        values = []
        vqe = VQE(ansatz3, optimizer, callback=store_intermediate_result, initial_point=params, 
                quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
        result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
        print()

        if (best_res3 is None) or (values[-1] < best_res3):
            best_res3 = values[-1]
            converge_cnts3[i] = np.asarray(counts)
            converge_vals3[i] = np.asarray(values)
print('\nOptimization complete      ')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts3, converge_vals3)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals3[i][-1], abs(ref - converge_vals3[i][-1]))
    pylab.plot(converge_cnts3[i], converge_vals3[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts3[i][:], abs(ref - converge_vals3[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals1[:, -1]))

In [ ]:
ansatz4 = TwoLocal(5, rotation_blocks=['ry'], entanglement_blocks='cx', entanglement='linear', reps=3).decompose()
print(ansatz4.depth())
ansatz4.draw(output='mpl', fold=-1)

In [ ]:
optimizers = [COBYLA(maxiter=8000), L_BFGS_B(maxfun=8000), SLSQP(maxiter=1000), NELDER_MEAD(maxfev=8000)]
# from qiskit.algorithms.optimizers import NELDER_MEAD
# optimizers = [SLSQP(maxiter=600)]
converge_cnts4 = np.empty([len(optimizers)], dtype=object)
converge_vals4 = np.empty([len(optimizers)], dtype=object)

repeat = 3
params = None
# params = np.array([0.0 for i in range(ansatz1.num_parameters)])
for i, optimizer in enumerate(optimizers):
    print('Optimizer: {}        '.format(type(optimizer).__name__))
    algorithm_globals.random_seed = 42

    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        print(f'\r{eval_count}, {mean}', end='')
    
    best_res4 = None
    for j in range(repeat):
        counts = []
        values = []
        vqe = VQE(ansatz4, optimizer, callback=store_intermediate_result, initial_point=params, 
                quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
        result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
        print()

        if (best_res4 is None) or (values[-1] < best_res4):
            best_res4 = values[-1]
            converge_cnts4[i] = np.asarray(counts)
            converge_vals4[i] = np.asarray(values)
print('\nOptimization complete      ')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts4, converge_vals4)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals4[i][-1], abs(ref - converge_vals4[i][-1]))
    pylab.plot(converge_cnts4[i], converge_vals4[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts4[i][:], abs(ref - converge_vals4[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals1[:, -1]))

In [ ]:
ansatz5 = TwoLocal(5, rotation_blocks=['ry'], entanglement_blocks='cx', entanglement='linear', reps=4).decompose()
print(ansatz5.depth())
ansatz5.draw(output='mpl', fold=-1)

In [ ]:
optimizers = [COBYLA(maxiter=8000), L_BFGS_B(maxfun=8000), SLSQP(maxiter=1000), NELDER_MEAD(maxfev=8000)]
# from qiskit.algorithms.optimizers import NELDER_MEAD
# optimizers = [SLSQP(maxiter=600)]
converge_cnts5 = np.empty([len(optimizers)], dtype=object)
converge_vals5 = np.empty([len(optimizers)], dtype=object)

repeat = 3
params = None
# params = np.array([0.0 for i in range(ansatz1.num_parameters)])
for i, optimizer in enumerate(optimizers):
    print('Optimizer: {}        '.format(type(optimizer).__name__))
    algorithm_globals.random_seed = 42

    def store_intermediate_result(eval_count, parameters, mean, std):
        counts.append(eval_count)
        values.append(mean)
        print(f'\r{eval_count}, {mean}', end='')
    
    best_res5 = None
    for j in range(repeat):
        counts = []
        values = []
        vqe = VQE(ansatz5, optimizer, callback=store_intermediate_result, initial_point=params, 
                quantum_instance=QuantumInstance(backend=Aer.get_backend('statevector_simulator')))
        result = vqe.compute_minimum_eigenvalue(operator=h_dvr_pauli)
        print()

        if (best_res5 is None) or (values[-1] < best_res5):
            best_res5 = values[-1]
            converge_cnts5[i] = np.asarray(counts)
            converge_vals5[i] = np.asarray(values)
print('\nOptimization complete      ')

In [ ]:
import pylab

ref = -115.26788457220933
# print(converge_cnts4, converge_vals4)
pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    print(type(optimizer).__name__, converge_vals5[i][-1], abs(ref - converge_vals5[i][-1]))
    pylab.plot(converge_cnts5[i], converge_vals5[i], label=type(optimizer).__name__)
# pylab.axhline(eigvals[0])

print(abs(ref - eigvals[0]))
pylab.axhline(eigvals[0], ls='--', c='blue', label='Numerical g.s.')
pylab.axhline(ref, ls='--', c='black', label='Converged g.s.')
pylab.xlabel('Eval count')
pylab.ylabel('Energy (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
# pylab.xlim(800, 1000)
pylab.ylim(-116, -80)
pylab.legend(loc='upper right');

In [ ]:
# pylab.rcParams['figure.figsize'] = (12, 8)
for i, optimizer in enumerate(optimizers):
    # pylab.plot(converge_cnts1[i][:], abs(ref - converge_vals1[i][:]), label=type(optimizer).__name__)
    pylab.plot(converge_cnts5[i][:], abs(ref - converge_vals5[i][:]), label=type(optimizer).__name__)
pylab.axhline(abs(ref - eigvals[0]), ls='--', c='blue', label='DVR ground state')
pylab.xlabel('Eval count')
pylab.ylabel('Energy difference from converged ground state (cm$^{-1}$)')
# pylab.title('Energy convergence for various optimizers')
pylab.yscale('log')
# plt.ylim(0,1)
pylab.legend(loc='upper right');
# print(abs(ref - converge_vals1[:, -1]))

In [ ]:
%autoreload
N_R = 35
N_theta = 4
l = 1
K_max = 1
r_min = 3.4
r_max = 5

Rs_DVR, Xs_K = get_DVR_Rtheta(N_R, N_theta, K_max, l=l, r_min=r_min, r_max=r_max)
Rs_angs = Rs_DVR * arhcl_params['S'] * au_to_angs
print(Rs_angs)
print(Rs_angs.shape)
N_R_lim = Rs_angs.shape[0]

h_dvr0 = get_ham_DVR(pot2d, N_R, N_theta, 0, K_max, l=l, r_min=r_min, r_max=r_max, trunc=0) * hartree
h_dvr1 = get_ham_DVR(pot2d, N_R, N_theta, 1, K_max, l=l, r_min=r_min, r_max=r_max, trunc=0) * hartree
print(h_dvr0.shape)
print(h_dvr1.shape)
eigvals0, eigvecs0 = np.linalg.eigh(h_dvr0)
eigvals1, eigvecs1 = np.linalg.eigh(h_dvr1)
print(eigvals0[0], eigvecs0.T[0])
print(eigvals1[0], eigvecs1.T[0])
plt.bar(range(eigvecs1.shape[1]), eigvecs1.T[0])
ref1 = -115.15276858268108

In [ ]:
cmap = plt.get_cmap('tab10')
theta_p, r_p, v_p = theta[v < 0], r[v < 0], v[v < 0]
coef = 180 / np.pi

p = plt.tricontour(theta_p[r_p < 5.5] * coef, r_p[r_p < 5.5], v_p[r_p < 5.5], levels=np.linspace(-175, 0, 26), colors='black', linestyles='solid', alpha=0.8)
# plt.plot(theta[ind_min1], r[ind_min1], 'x', c='black')
# plt.plot(theta[ind_min2], r[ind_min2], 'x', c='black')
# plt.plot(1.8754889808102941, 3.897, 'o', c='black')
Vs_K = []
for K in range(K_max + 1):
    Rs_grid, Xs_grid = np.meshgrid(Rs_DVR * arhcl_params['S'] * au_to_angs, Xs_K[K])
    Rs_grid, Xs_grid = Rs_grid.flatten(), Xs_grid.flatten()
    Vs = pot2d(Rs_grid, Xs_grid)
    Vs_K.append(Vs)
Vs_K = np.array(Vs_K)

labels = p.levels[:-15:]
plt.clabel(p, inline=1, fontsize=5, levels=labels, fmt='%.2f', use_clabeltext=True)
plt.ylim()
plt.xlabel('$\\theta (\degree)$')
plt.ylabel('$R$ (A)')
plt.ylim((3, 5.5))
# plt.xlim((0, np.pi))

# offset = 0.05
# plt.bar(np.arccos(Xs_K[0]), eigvecs5.T[0][::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 1 * offset, eigvecs5.T[0][1::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 2 * offset, eigvecs5.T[0][2::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 3 * offset, eigvecs5.T[0][3::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.bar(np.arccos(Xs_K[0]) + 4 * offset, eigvecs5.T[0][4::5] * 1.5, bottom=4, width=0.05, color='black', alpha=0.4)
# plt.axhline(4.0, linestyle='dashed', c='black', linewidth=0.2)

for K in range(K_max + 1):
    for i in range(N_theta):
        evec = eigvecs.T[0][i * N_R_lim:(i+1) * N_R_lim]
        epos = np.where(evec >=0, evec, 0)
        eneg = np.where(evec <0, -evec, 0)
        plt.scatter(np.repeat(np.arccos(Xs_K[K][i]) * coef, N_R_lim), Rs_DVR * arhcl_params['S'] * au_to_angs, marker='o', c='black', zorder=20, alpha=0.7)
        # plt.scatter(np.repeat(np.arccos(Xs_K[0][i]), N_R_lim), Rs_DVR * arhcl_params['S'] * au_to_angs, marker='o', facecolors='black', edgecolors='black', s=epos * 400, zorder=20, alpha=0.7)
        # plt.scatter(np.repeat(np.arccos(Xs_K[0][i]), N_R_lim), Rs_DVR * arhcl_params['S'] * au_to_angs, facecolors='none', edgecolors='black', s=eneg * 400, alpha=0.7, zorder=20)

In [ ]:
h_dvr_pauli0 = pauli_decompose(h_dvr0)
h_dvr_pauli1 = pauli_decompose(h_dvr1)

np.allclose(h_dvr0, h_dvr_pauli0.to_matrix())
np.allclose(h_dvr1, h_dvr_pauli1.to_matrix())

print(h_dvr0.shape)
print(h_dvr_pauli0.coeffs.shape)
print(h_dvr1.shape)
print(h_dvr_pauli1.coeffs.shape)
# print(h_dvr_pauli)